In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate sentencepiece


Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 77.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 103.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:

!pip install numpy
!pip install -U "jax[cpu]"   # CPU-only JAX
!pip install torch --index-url https://download.pytorch.org/whl/cu118
!pip install openai
!pip install tiktoken
!pip install tqdm
!pip install matplotlib
!pip install "pandas<2.0.0"     # LSTPrompt repo limitation
!pip install darts
!pip install gpytorch
!pip install transformers
!pip install datasets
!pip install multiprocess
!pip install sentencepiece
!pip install accelerate
!pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 77.1 MB/s eta 0:00:00
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.5.1
    Uninstalling jaxlib-0.5.1:
      Successfully uninstalled jaxlib-0.5.1
  Attempting uninstall: jax
    Found existing installation: jax 0.5.2
    Uninstalling jax-0.5.2:
      Successfully uninstalled jax-0.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
Looking in indexes: https://downl

In [ ]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.97.1
    Uninstalling openai-1.97.1:
      Successfully uninstalled openai-1.97.1


In [ ]:
!git clone https://github.com/AdityaLab/lstprompt.git
%cd /content/lstprompt

Cloning into 'lstprompt'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 81 (delta 13), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 385.71 KiB | 11.69 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/lstprompt


In [ ]:
import math
import requests
import os
os.environ['OMP_NUM_THREADS'] = '4'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openai
import time
import re
# Load full forecast and attention files
forecast_df = pd.read_csv("/content/tft_median_and_quantiles_forecast_sample_0.csv", header=None, skiprows=1)
forecast_df = forecast_df[0].str.split(";", expand=True)
forecast_df.columns = ["horizon", "median", "l90", "u90", "l95", "u95"]
forecast_df = forecast_df.astype({"horizon": float, "median": float, "l90": float, "u90": float})


attn_df = pd.read_csv("/content/tft_attention_weights_sample_0 (1).csv", sep="\t", index_col=0)

# Now all values should be numeric, and index is like 'prediction_horizon_1', etc.
attn_df = attn_df.astype(float)

print(f"✅ attn_df shape: {attn_df.shape}")

print(f"🔍 forecast_df rows: {len(forecast_df)}")
print(f"🔍 attn_df rows: {len(attn_df)}")


# GEMINI setup
API_KEY = API_KEY
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={API_KEY}"
headers = { "Content-Type": "application/json" }

# Store improved medians
improved_medians = []

# Loop over forecast in 5-minute chunks
for start in range(0, 1440, 12):
    end = min(start + 12, 1440)
    prompt_blocks = []

    for i in range(start, end):
        row = forecast_df.iloc[i]
        attn = attn_df.iloc[i].values
        attn_str = " ".join([f"{float(a):.5f}" for a in attn])
        block = (
            f"Minute Horizon: {int(row['horizon'])}\n"
            f"Median: {row['median']:.2f}\n"
            f"90% Confidence: [{row['l90']:.2f}, {row['u90']:.2f}]\n"
            f"Encoder Attention: {attn_str}\n"
        )
        prompt_blocks.append(block)

    instruction = (
        "Improve the median values using all the encoder steps and 90% confidence intervals.\n"
        "Only output the 12 improved median values in order, one per line. No explanations.\n"
    )
    prompt = instruction + "\n".join(prompt_blocks)

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ]
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        try:
            content = response.json()
            reply_text = content["candidates"][0]["content"]["parts"][0]["text"]

            # Parse 5 numbers from Gemini output
            numbers = re.findall(r"[-+]?\d*\.\d+|\d+", reply_text)
            numbers = [float(n) for n in numbers]

# Optionally warn if the number of values is wrong
            if len(numbers) != (end - start):
              print(f"⚠️ Expected {end-start} values, but got {len(numbers)}. Raw reply:\n{reply_text}")
            else:
              improved_medians.extend(numbers)
              print(f"✅ Processed minutes {start + 1} to {end} → {numbers}")
            improved_medians.extend(numbers)

            time.sleep(10)
        except Exception as e:
            print(f"⚠️ Failed to parse output for block {start}-{end}:", e)
    else:
        print(f"❌ API Error for block {start}-{end}: {response.status_code} {response.text}")
        break  # Optional: stop if rate-limited or error
    # === Export to CSV ===
print(f"✅ Improved medians collected: {len(improved_medians)}")

output_df = forecast_df.copy()
output_df["improved_median"] = improved_medians

# Save to CSV
output_path = "/content/improved_medians.csv"
output_df.to_csv(output_path, index=False)
print(f"📁 Saved improved forecast to: {output_path}")


✅ attn_df shape: (1440, 0)
🔍 forecast_df rows: 1440
🔍 attn_df rows: 1440
✅ Processed minutes 1 to 12 → [240.315, 240.22, 240.215, 240.19, 240.16, 240.13, 240.085, 240.04, 240.005, 239.95, 239.97, 240.055]
✅ Processed minutes 13 to 24 → [240.685, 240.8, 240.925, 241.045, 241.13, 241.18, 241.22, 241.245, 241.28, 241.31, 241.35, 241.4]
✅ Processed minutes 25 to 36 → [241.31, 241.36, 241.4, 241.43, 241.44, 241.45, 241.46, 241.47, 241.49, 241.51, 241.52, 241.54]
✅ Processed minutes 37 to 48 → [241.16, 241.16, 241.17, 241.18, 241.18, 241.18, 241.19, 241.2, 241.23, 241.27, 241.29, 241.3]
✅ Processed minutes 49 to 60 → [241.26, 241.26, 241.25, 241.24, 241.23, 241.22, 241.21, 241.2, 241.19, 241.18, 241.17, 241.42]
✅ Processed minutes 61 to 72 → [241.49, 241.54, 241.57, 241.6, 241.61, 241.62, 241.62, 241.64, 241.66, 241.7, 241.76, 241.82]
✅ Processed minutes 73 to 84 → [241.83, 241.835, 241.825, 241.805, 241.775, 241.72, 241.69, 241.66, 241.63, 241.6, 241.585, 241.575]
✅ Processed minutes 85 to 

ValueError: Length of values (2880) does not match length of index (1440)

In [ ]:
import re

text = """✅ Processed minutes 1 to 12 → [240.315, 240.22, 240.215, 240.19, 240.16, 240.13, 240.085, 240.04, 240.005, 239.95, 239.97, 240.055]
✅ Processed minutes 13 to 24 → [240.685, 240.8, 240.925, 241.045, 241.13, 241.18, 241.22, 241.245, 241.28, 241.31, 241.35, 241.4]
✅ Processed minutes 25 to 36 → [241.31, 241.36, 241.4, 241.43, 241.44, 241.45, 241.46, 241.47, 241.49, 241.51, 241.52, 241.54]
✅ Processed minutes 37 to 48 → [241.16, 241.16, 241.17, 241.18, 241.18, 241.18, 241.19, 241.2, 241.23, 241.27, 241.29, 241.3]
✅ Processed minutes 49 to 60 → [241.26, 241.26, 241.25, 241.24, 241.23, 241.22, 241.21, 241.2, 241.19, 241.18, 241.17, 241.42]
✅ Processed minutes 61 to 72 → [241.49, 241.54, 241.57, 241.6, 241.61, 241.62, 241.62, 241.64, 241.66, 241.7, 241.76, 241.82]
✅ Processed minutes 73 to 84 → [241.83, 241.835, 241.825, 241.805, 241.775, 241.72, 241.69, 241.66, 241.63, 241.6, 241.585, 241.575]
✅ Processed minutes 85 to 96 → [241.59, 241.58, 241.58, 241.58, 241.59, 241.6, 241.61, 241.62, 241.63, 241.64, 241.64, 241.65]
✅ Processed minutes 97 to 108 → [241.65, 241.65, 241.65, 241.66, 241.67, 241.7, 241.74, 241.79, 241.82, 241.85, 241.87, 241.88]
✅ Processed minutes 109 to 120 → [241.85, 241.86, 241.87, 241.875, 241.88, 241.88, 241.885, 241.89, 241.9, 241.91, 241.92, 242.115]
✅ Processed minutes 121 to 132 → [241.37, 241.46, 241.54, 241.61, 241.67, 241.71, 241.75, 241.79, 241.83, 241.86, 241.88, 241.89]
✅ Processed minutes 133 to 144 → [243.2, 243.19, 243.17, 243.15, 243.15, 243.17, 243.19, 243.2, 243.21, 243.22, 243.22, 243.22]
✅ Processed minutes 145 to 156 → [242.74, 242.73, 242.74, 242.75, 242.78, 242.82, 242.86, 242.91, 242.96, 243.01, 243.05, 243.1]
✅ Processed minutes 157 to 168 → [243.62, 243.66, 243.7, 243.73, 243.77, 243.81, 243.86, 243.9, 243.94, 243.97, 243.99, 244.0]
✅ Processed minutes 169 to 180 → [243.48, 243.47, 243.47, 243.46, 243.45, 243.44, 243.43, 243.42, 243.41, 243.4, 243.4, 243.48]
✅ Processed minutes 181 to 192 → [243.89, 243.87, 243.83, 243.79, 243.74, 243.69, 243.65, 243.62, 243.6, 243.59, 243.58, 243.57]
✅ Processed minutes 193 to 204 → [243.645, 243.685, 243.74, 243.755, 243.715, 243.71, 243.735, 243.76, 243.78, 243.8, 243.82, 243.83]
✅ Processed minutes 205 to 216 → [243.34, 243.36, 243.37, 243.39, 243.41, 243.43, 243.44, 243.45, 243.45, 243.45, 243.45, 243.45]
✅ Processed minutes 217 to 228 → [243.44, 243.44, 243.44, 243.44, 243.44, 243.45, 243.46, 243.47, 243.48, 243.49, 243.49, 243.49]
✅ Processed minutes 229 to 240 → [243.52, 243.51, 243.5, 243.48, 243.47, 243.45, 243.43, 243.42, 243.41, 243.4, 243.39, 243.55]
✅ Processed minutes 241 to 252 → [243.53, 243.53, 243.53, 243.53, 243.52, 243.51, 243.5, 243.49, 243.48, 243.48, 243.48, 243.47]
✅ Processed minutes 253 to 264 → [243.49, 243.52, 243.57, 243.62, 243.62, 243.63, 243.63, 243.63, 243.63, 243.62, 243.61, 243.6]
✅ Processed minutes 265 to 276 → [243.59, 243.57, 243.56, 243.54, 243.53, 243.51, 243.5, 243.47, 243.44, 243.4, 243.35, 243.29]
✅ Processed minutes 277 to 288 → [243.21, 243.12, 243.02, 242.91, 242.82, 242.75, 242.69, 242.65, 242.63, 242.6, 242.58, 242.51]
✅ Processed minutes 289 to 300 → [242.4, 242.23, 242.03, 241.91, 241.87, 241.87, 241.86, 241.83, 241.76, 241.66, 241.54, 241.41]
✅ Processed minutes 301 to 312 → [241.36, 241.3, 241.24, 241.19, 241.14, 241.09, 241.04, 240.99, 240.94, 240.92, 240.89, 240.87]
✅ Processed minutes 313 to 324 → [240.73, 240.67, 240.61, 240.56, 240.54, 240.54, 240.55, 240.55, 240.54, 240.54, 240.55, 240.56]
✅ Processed minutes 325 to 336 → [240.575, 240.57, 240.575, 240.57, 240.565, 240.555, 240.545, 240.535, 240.525, 240.505, 240.49, 240.465]
✅ Processed minutes 337 to 348 → [240.45, 240.43, 240.42, 240.41, 240.4, 240.41, 240.42, 240.43, 240.44, 240.43, 240.41, 240.38]
✅ Processed minutes 349 to 360 → [240.74, 240.71, 240.67, 240.63, 240.6, 240.56, 240.52, 240.49, 240.46, 240.44, 240.41, 240.38]
✅ Processed minutes 361 to 372 → [240.515, 240.45, 240.39, 240.335, 240.32, 240.27, 240.22, 240.165, 240.12, 240.07, 240.015, 239.95]
✅ Processed minutes 373 to 384 → [239.37, 239.27, 239.17, 239.1, 239.07, 239.06, 239.05, 239.05, 239.04, 239.04, 239.04, 239.04]
✅ Processed minutes 385 to 396 → [239.55, 239.55, 239.55, 239.56, 239.58, 239.61, 239.64, 239.68, 239.71, 239.74, 239.755, 239.77]
✅ Processed minutes 397 to 408 → [239.785, 239.805, 239.825, 239.85, 239.885, 239.92, 239.95, 239.955, 239.935, 239.895, 239.855, 239.82]
✅ Processed minutes 409 to 420 → [239.79, 239.77, 239.75, 239.73, 239.71, 239.7, 239.69, 239.68, 239.67, 239.66, 239.65, 239.43]
✅ Processed minutes 421 to 432 → [238.965, 238.975, 239.0, 239.025, 239.045, 239.035, 239.035, 239.035, 239.03, 239.02, 239.015, 238.995]
✅ Processed minutes 433 to 444 → [239.455, 239.41, 239.405, 239.43, 239.445, 239.455, 239.46, 239.46, 239.46, 239.46, 239.47, 239.47]
✅ Processed minutes 445 to 456 → [239.27, 239.28, 239.3, 239.33, 239.36, 239.4, 239.44, 239.48, 239.51, 239.53, 239.55, 239.56]
✅ Processed minutes 457 to 468 → [239.84, 239.86, 239.88, 239.92, 239.95, 240.0, 240.02, 240.03, 240.01, 239.95, 239.89, 239.83]
✅ Processed minutes 469 to 480 → [239.85, 239.83, 239.8, 239.79, 239.77, 239.76, 239.75, 239.74, 239.73, 239.72, 239.71, 239.35]
✅ Processed minutes 481 to 492 → [238.86, 238.81, 238.76, 238.7, 238.66, 238.74, 238.89, 239.29, 239.92, 240.28, 240.45, 240.53]
✅ Processed minutes 493 to 504 → [240.78, 240.85, 240.9, 240.93, 240.95, 240.97, 240.98, 241.0, 241.0, 241.01, 241.0, 240.99]
✅ Processed minutes 505 to 516 → [240.97, 240.94, 240.92, 240.91, 240.91, 240.92, 240.93, 240.94, 240.94, 240.94, 240.94, 240.93]
✅ Processed minutes 517 to 528 → [240.48, 240.47, 240.46, 240.45, 240.44, 240.43, 240.42, 240.38, 240.33, 240.28, 240.24, 240.23]
✅ Processed minutes 529 to 540 → [240.735, 240.775, 240.84, 240.915, 240.985, 241.05, 241.1, 241.14, 241.17, 241.185, 241.195, 240.76]
✅ Processed minutes 541 to 552 → [240.265, 240.29, 240.315, 240.335, 240.35, 240.365, 240.375, 240.385, 240.405, 240.425, 240.455, 240.49]
✅ Processed minutes 553 to 564 → [240.545, 240.615, 240.66, 240.685, 240.695, 240.695, 240.7, 240.7, 240.695, 240.69, 240.68, 240.665]
✅ Processed minutes 565 to 576 → [240.645, 240.615, 240.585, 240.56, 240.54, 240.52, 240.495, 240.465, 240.435, 240.405, 240.37, 240.34]
✅ Processed minutes 577 to 588 → [240.32, 240.29, 240.25, 240.25, 240.23, 240.21, 240.19, 240.16, 240.11, 240.07, 240.05, 240.06]
✅ Processed minutes 589 to 600 → [240.18, 240.21, 240.24, 240.28, 240.32, 240.36, 240.41, 240.45, 240.49, 240.53, 240.56, 240.11]
✅ Processed minutes 601 to 612 → [240.55, 240.56, 240.56, 240.56, 240.55, 240.55, 240.54, 240.53, 240.54, 240.54, 240.55, 240.57]
✅ Processed minutes 613 to 624 → [240.625, 240.72, 240.775, 240.77, 240.75, 240.735, 240.725, 240.72, 240.705, 240.715, 240.705, 240.695]
✅ Processed minutes 625 to 636 → [240.18, 240.17, 240.15, 240.14, 240.15, 240.18, 240.23, 240.29, 240.35, 240.42, 240.49, 240.56]
✅ Processed minutes 637 to 648 → [241.11, 241.2, 241.28, 241.38, 241.47, 241.56, 241.64, 241.7, 241.73, 241.75, 241.75, 241.75]
✅ Processed minutes 649 to 660 → [240.95, 240.95, 240.95, 240.95, 240.94, 240.93, 240.93, 240.92, 240.91, 240.9, 240.89, 241.29]
✅ Processed minutes 661 to 672 → [241.24, 241.21, 241.19, 241.22, 241.32, 241.41, 241.53, 241.66, 241.82, 241.99, 242.2, 242.43]
✅ Processed minutes 673 to 684 → [242.79, 242.98, 243.12, 243.19, 243.21, 243.2, 243.19, 243.19, 243.21, 243.23, 243.24, 243.24]
✅ Processed minutes 685 to 696 → [243.23, 243.22, 243.21, 243.2, 243.2, 243.21, 243.24, 243.27, 243.33, 243.39, 243.44, 243.47]
✅ Processed minutes 697 to 708 → [242.98, 242.99, 242.99, 243.0, 243.0, 243.0, 243.0, 242.99, 242.98, 242.96, 242.94, 242.92]
✅ Processed minutes 709 to 720 → [242.91, 242.93, 242.96, 242.99, 243.02, 243.04, 243.07, 243.09, 243.11, 243.12, 243.12, 243.14]
✅ Processed minutes 721 to 732 → [242.98, 242.9, 242.86, 242.88, 243.04, 243.31, 243.55, 243.64, 243.64, 243.71, 243.79, 243.88]
✅ Processed minutes 733 to 744 → [244.36, 244.44, 244.52, 244.59, 244.64, 244.68, 244.72, 244.75, 244.78, 244.78, 244.78, 244.77]
✅ Processed minutes 745 to 756 → [244.36, 244.35, 244.34, 244.33, 244.32, 244.31, 244.3, 244.29, 244.28, 244.28, 244.27, 244.27]
✅ Processed minutes 757 to 768 → [244.21, 244.2, 244.2, 244.21, 244.23, 244.27, 244.32, 244.37, 244.4, 244.37, 244.34, 244.31]
✅ Processed minutes 769 to 780 → [244.24, 244.23, 244.22, 244.21, 244.19, 244.18, 244.16, 244.15, 244.14, 244.13, 244.12, 244.09]
✅ Processed minutes 781 to 792 → [244.185, 244.24, 244.26, 244.28, 244.295, 244.315, 244.33, 244.35, 244.365, 244.385, 244.42, 244.46]
✅ Processed minutes 793 to 804 → [244.86, 244.9, 244.94, 244.98, 245.01, 245.05, 245.08, 245.11, 245.15, 245.18, 245.21, 245.24]
✅ Processed minutes 805 to 816 → [244.94, 244.97, 245.0, 245.03, 245.05, 245.05, 245.04, 245.02, 244.99, 244.96, 244.92, 244.89]
✅ Processed minutes 817 to 828 → [244.86, 244.83, 244.81, 244.81, 244.82, 244.86, 244.91, 244.97, 245.03, 245.09, 245.14, 245.18]
✅ Processed minutes 829 to 840 → [245.55, 245.57, 245.58, 245.57, 245.55, 245.52, 245.48, 245.43, 245.38, 245.34, 245.29, 245.5]
✅ Processed minutes 841 to 852 → [245.78, 245.82, 245.84, 245.85, 245.87, 245.88, 245.9, 245.91, 245.92, 245.93, 245.95, 245.97]
✅ Processed minutes 853 to 864 → [245.51, 245.53, 245.53, 245.53, 245.52, 245.52, 245.51, 245.5, 245.5, 245.49, 245.48, 245.47]
✅ Processed minutes 865 to 876 → [245.47, 245.46, 245.45, 245.44, 245.43, 245.42, 245.41, 245.4, 245.38, 245.37, 245.36, 245.35]
✅ Processed minutes 877 to 888 → [245.33, 245.32, 245.31, 245.31, 245.31, 245.32, 245.33, 245.34, 245.35, 245.35, 245.36, 245.35]
✅ Processed minutes 889 to 900 → [245.34, 245.33, 245.32, 245.31, 245.3, 245.29, 245.28, 245.26, 245.25, 245.24, 245.23, 244.99]
✅ Processed minutes 901 to 912 → [245.32, 245.3, 245.27, 245.25, 245.23, 245.21, 245.19, 245.17, 245.15, 245.15, 245.15, 245.16]
✅ Processed minutes 913 to 924 → [244.87, 244.87, 244.85, 244.83, 244.82, 244.8, 244.79, 244.78, 244.77, 244.76, 244.75, 244.74]
✅ Processed minutes 925 to 936 → [244.72, 244.71, 244.69, 244.68, 244.67, 244.66, 244.66, 244.66, 244.66, 244.66, 244.65, 244.65]
✅ Processed minutes 937 to 948 → [244.65, 244.65, 244.65, 244.66, 244.66, 244.67, 244.67, 244.68, 244.68, 244.68, 244.68, 244.68]
✅ Processed minutes 949 to 960 → [244.665, 244.665, 244.665, 244.665, 244.655, 244.655, 244.65, 244.645, 244.645, 244.635, 244.635, 244.44]
✅ Processed minutes 961 to 972 → [244.705, 244.69, 244.68, 244.675, 244.675, 244.665, 244.665, 244.66, 244.65, 244.65, 244.65, 244.66]
✅ Processed minutes 973 to 984 → [244.37, 244.37, 244.34, 244.32, 244.29, 244.28, 244.27, 244.28, 244.28, 244.29, 244.29, 244.29]
✅ Processed minutes 985 to 996 → [244.64, 244.63, 244.62, 244.61, 244.6, 244.6, 244.6, 244.6, 244.61, 244.62, 244.63, 244.65]
✅ Processed minutes 997 to 1008 → [244.68, 244.69, 244.7, 244.72, 244.73, 244.74, 244.75, 244.76, 244.76, 244.77, 244.78, 244.78]
✅ Processed minutes 1009 to 1020 → [244.41, 244.41, 244.41, 244.4, 244.4, 244.39, 244.38, 244.37, 244.37, 244.36, 244.35, 244.19]
✅ Processed minutes 1021 to 1032 → [244.18, 244.17, 244.17, 244.17, 244.17, 244.17, 244.16, 244.14, 244.13, 244.12, 244.12, 244.13]
✅ Processed minutes 1033 to 1044 → [244.135, 244.14, 244.135, 244.125, 244.1, 244.04, 243.965, 243.875, 243.775, 243.685, 243.615, 243.55]
✅ Processed minutes 1045 to 1056 → [243.94, 243.9, 243.9, 243.94, 243.97, 244.0, 244.06, 244.11, 244.16, 244.21, 244.27, 244.31]
✅ Processed minutes 1057 to 1068 → [244.19, 244.21, 244.24, 244.26, 244.29, 244.32, 244.33, 244.36, 244.36, 244.38, 244.39, 244.4]
✅ Processed minutes 1069 to 1080 → [244.13, 244.14, 244.14, 244.14, 244.14, 244.13, 244.12, 244.115, 244.105, 244.095, 244.085, 244.07]
✅ Processed minutes 1081 to 1092 → [244.09, 244.09, 244.1, 244.1, 244.1, 244.09, 244.09, 244.08, 244.07, 244.06, 244.06, 244.08]
✅ Processed minutes 1093 to 1104 → [244.1, 244.13, 244.16, 244.18, 244.18, 244.17, 244.17, 244.16, 244.16, 244.16, 244.16, 244.16]
✅ Processed minutes 1105 to 1116 → [244.155, 244.16, 244.17, 244.18, 244.2, 244.215, 244.23, 244.245, 244.255, 244.265, 244.275, 244.28]
✅ Processed minutes 1117 to 1128 → [244.28, 244.29, 244.29, 244.29, 244.3, 244.32, 244.33, 244.34, 244.35, 244.35, 244.34, 244.33]
✅ Processed minutes 1129 to 1140 → [244.36, 244.35, 244.34, 244.33, 244.32, 244.31, 244.3, 244.29, 244.28, 244.27, 244.26, 244.2]
✅ Processed minutes 1141 to 1152 → [244.16, 244.16, 244.16, 244.14, 244.09, 243.94, 243.65, 243.2, 242.49, 241.78, 241.29, 241.0]
✅ Processed minutes 1153 to 1164 → [240.76, 240.63, 240.54, 240.52, 240.51, 240.5, 240.5, 240.51, 240.56, 240.63, 240.7, 240.77]
✅ Processed minutes 1165 to 1176 → [240.83, 240.87, 240.9, 240.92, 240.95, 240.98, 241.01, 241.04, 241.08, 241.11, 241.12, 241.14]
✅ Processed minutes 1177 to 1188 → [240.97, 240.85, 240.69, 240.54, 240.38, 240.26, 240.16, 240.06, 239.96, 239.87, 239.79, 239.75]
✅ Processed minutes 1189 to 1200 → [239.72, 239.7, 239.68, 239.66, 239.65, 239.63, 239.61, 239.6, 239.58, 239.58, 239.58, 239.17]
✅ Processed minutes 1201 to 1212 → [239.62, 239.59, 239.57, 239.55, 239.54, 239.54, 239.56, 239.59, 239.63, 239.68, 239.74, 239.81]
✅ Processed minutes 1213 to 1224 → [239.915, 240.055, 240.205, 240.345, 240.445, 240.525, 240.6, 240.675, 240.755, 240.835, 240.91, 240.97]
✅ Processed minutes 1225 to 1236 → [240.51, 240.52, 240.53, 240.54, 240.55, 240.56, 240.57, 240.59, 240.62, 240.65, 240.69, 240.72]
✅ Processed minutes 1237 to 1248 → [240.74, 240.77, 240.79, 240.82, 240.84, 240.85, 240.84, 240.81, 240.76, 240.71, 240.65, 240.61]
✅ Processed minutes 1249 to 1260 → [240.54, 240.52, 240.48, 240.46, 240.43, 240.41, 240.38, 240.35, 240.33, 240.31, 240.28, 239.94]
✅ Processed minutes 1261 to 1272 → [239.89, 239.84, 239.79, 239.75, 239.73, 239.73, 239.74, 239.76, 239.77, 239.79, 239.87, 239.93]
✅ Processed minutes 1273 to 1284 → [239.935, 239.86, 239.75, 239.675, 239.695, 239.78, 239.88, 239.96, 240.03, 240.04, 240.04, 240.02]
✅ Processed minutes 1285 to 1296 → [239.99, 239.95, 239.91, 239.89, 239.88, 239.88, 239.9, 239.94, 240.0, 240.07, 240.14, 240.21]
✅ Processed minutes 1297 to 1308 → [240.24, 240.3, 240.35, 240.4, 240.45, 240.48, 240.48, 240.43, 240.33, 240.2, 240.09, 239.98]
✅ Processed minutes 1309 to 1320 → [239.93, 239.86, 239.79, 239.74, 239.68, 239.63, 239.59, 239.55, 239.53, 239.53, 239.53, 239.34]
✅ Processed minutes 1321 to 1332 → [239.815, 239.785, 239.77, 239.76, 239.76, 239.785, 239.825, 239.865, 239.92, 239.99, 240.08, 240.185]
✅ Processed minutes 1333 to 1344 → [239.74, 239.74, 239.73, 239.69, 239.64, 239.62, 239.62, 239.65, 239.68, 239.71, 239.75, 239.79]
✅ Processed minutes 1345 to 1356 → [239.82, 239.86, 239.89, 239.92, 239.93, 239.94, 239.96, 239.98, 240.0, 240.02, 240.03, 240.04]
✅ Processed minutes 1357 to 1368 → [240.05, 240.05, 240.06, 240.07, 240.1, 240.13, 240.17, 240.21, 240.23, 240.24, 240.23, 240.21]
✅ Processed minutes 1369 to 1380 → [240.23, 240.21, 240.2, 240.18, 240.17, 240.16, 240.14, 240.14, 240.13, 240.13, 240.13, 239.69]
✅ Processed minutes 1381 to 1392 → [239.89, 239.89, 239.89, 239.91, 239.92, 239.93, 239.96, 239.98, 240.01, 240.03, 240.08, 240.09]
✅ Processed minutes 1393 to 1404 → [240.12, 240.15, 240.16, 240.2, 240.24, 240.28, 240.31, 240.33, 240.35, 240.37, 240.39, 240.43]
✅ Processed minutes 1405 to 1416 → [240.44, 240.49, 240.55, 240.62, 240.68, 240.74, 240.79, 240.85, 240.9, 240.96, 241.0, 241.03]
✅ Processed minutes 1417 to 1428 → [240.895, 240.915, 240.94, 240.96, 240.985, 241.035, 241.08, 241.09, 241.105, 241.115, 241.115, 241.115]
✅ Processed minutes 1429 to 1440 → [241.14, 241.14, 241.14, 241.13, 241.12, 241.09, 241.06, 241.04, 241.0, 240.97, 240.95, 240.16]
"""

# Find all occurrences of brackets with numbers inside
matches = re.findall(r'\[([^\]]+)\]', text)

# matches is a list of strings like "240.315, 240.22, 240.215, ..."
# Now split each string by comma and convert to float
values = []
for match in matches:
    nums = [float(x.strip()) for x in match.split(',')]
    values.extend(nums)

print(f"Total values extracted: {len(values)}")
# values is now a list of all floats in order

# Optionally verify length
assert len(values) == 1440, f"Expected 1440 values but got {len(values)}"
print(min(values))
print(max(values))
print(sum(values) / len(values))
import numpy as np



std_dev = np.std(values)
q25 = np.percentile(values, 25)
q50 = np.percentile(values, 50)  # also the median
q75 = np.percentile(values, 75)

print(f"Standard Deviation: {std_dev}")
print(f"25% Quartile: {q25}")
print(f"50% (Median): {q50}")
print(f"75% Quartile: {q75}")



Total values extracted: 1440
238.66
245.97
242.11671180555544
Standard Deviation: 1.9821151555499064
25% Quartile: 240.38
50% (Median): 241.5775
75% Quartile: 244.14
